# System Demo

Live demo of the classroom attendance system. The pipeline is implemented in `live_system_demo.py`.

The user can choose between three operating modes:
1. **Enrollment**: register a new user (features of all the accepted samples are stored, never the images)
2. **Identification**: open-set 1:N search with a reject option for users who are not enrolled
3. **Verification**: 1:1 check of a claimed identity

The liveness (replay-attack) detector can be switched on or off. At most 3 samples are used per session, and the
processing time of every stage is printed at the end.

In [1]:
%load_ext autoreload
%autoreload 2
# Make the face_auth modules importable and run from the repository root
# (relative paths such as models/ and Dataset/ are resolved from there)
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "face_auth"))

import os
import torch

import 02_system_demo as live_demo

SyntaxError: invalid decimal literal (907723123.py, line 13)

## Config

In [ ]:
# Operating thresholds calibrated in 01_face_authentication.ipynb (Tf, Ti, Tv, MagFace quality)
config = dict(live_demo.DEFAULT_CONFIG)
# config["quality_threshold"] = ...   # override any value here if needed
print({k: config[k] for k in ("Tf", "Ti", "Tv", "quality_threshold", "max_samples")})

TESTING_DIR = "Test"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
magface, liveness_model = live_demo.load_models(config, device, use_liveness=True)

## User Config

In [ ]:
MODE = "Identification"       # Options: "Enrollment", "Identification", "Verification"
USE_LIVENESS = True           # Activate / deactivate the liveness detector
USER_NAME = "new_user"        # Enrollment: name of the new user (also the folder with the samples)
CLAIMED_ID = 10               # Verification: claimed identity ID
DATA_DIR = os.path.join(TESTING_DIR, USER_NAME)

## Operating Flow

1. **Acquisition & detection**: at most `max_samples` (3) images are read from `DATA_DIR` and MTCNN crops the face in each one.
2. **Liveness detection** (only if `USE_LIVENESS = True`): every face gets a liveness score (0 = fake, 1 = live).
   Samples below `Tf` are discarded as replay attacks.
3. **Quality gate (acquisition module)**: MagFace scores every remaining sample, and samples that do not exceed the
   quality threshold are discarded. If **no** sample passes, the session stops and the user is asked to
   **re-acquire** the sample, as a real system would.
4. **Feature extraction**: InceptionResnetV1-VGGFace2 embeddings, L2-normalised.
   - *Enrollment* stores the features of **all** accepted samples.
   - *Identification / Verification* use only the **best-quality** sample.
5. **Matching**
   - *Verification*: the probe is matched **only** against the templates of the claimed ID, and the output is
     the matching score plus a MATCH / REJECT decision (`Tv`).
   - *Identification*: the output is the list of enrolled users whose score is above `Ti`, ordered by matching score.
     An empty list means the user is not recognized (open-set reject).

In [ ]:
try:
    session = live_demo.run_session(
        MODE, DATA_DIR, config["embedding_model"], device, magface, liveness_model,
        use_liveness=USE_LIVENESS, user_name=USER_NAME, claimed_id=CLAIMED_ID, config=config, show=True)
except live_demo.AcquisitionError as e:
    print(f"\n>> RE-ACQUISITION REQUIRED: {e}")
except ValueError as e:
    print(f"\n>> ERROR: {e}")